# MiniMax H3：CPU 提前准备
先选 **运行时 → 更改运行时类型 → CPU / 无硬件加速**，Python 3.12。这里不需要 GPU。
本笔记本将 Linux CUDA 安装包、精确依赖清单和模型权重保存在你的 Google Drive。
单任务权重约 **144 GB（134.2 GiB）**，另需安装包和余量，建议 Drive 可用空间 **180 GB 以上**。
CPU 会话可以长时间下载；中断后重新执行可利用 Hugging Face 已完成的缓存。
不要假设切换 GPU 后 `/content` 和已安装的库仍在。

In [ ]:
# 已填写当前仓库地址；如 fork 到其他账号，请修改此项。
REPO_URL = "https://github.com/simonFelix-Ai/colab.git"
REPO_REF = "main"  # 建议确认后改成自己的提交 SHA
PERSIST_ROOT = "/content/drive/MyDrive/minimax-h3-colab"
WORKFLOW = "t2va"  # t2va / fl2va 共用权重；ref2va 需要另一个 transformer

In [ ]:
import os, subprocess, sys
from pathlib import Path
assert REPO_URL.startswith("https://github.com/"), "请填写有效的 GitHub 仓库 URL"
assert sys.version_info[:2] == (3, 12), "本项目使用 Python 3.12，请切到兼容的 Colab 运行时"
PROJECT = Path("/content/minimax-h3-colab")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(PROJECT)], check=True)
else:
    assert subprocess.check_output(["git", "-C", str(PROJECT), "remote", "get-url", "origin"], text=True).strip() == REPO_URL, "现有目录属于其他仓库"
subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(PROJECT)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("代码版本:", COMMIT)
def run(*args):
    subprocess.run([str(x) for x in args], check=True, cwd=PROJECT)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
PERSIST = Path(PERSIST_ROOT)
PERSIST.mkdir(parents=True, exist_ok=True)
import hashlib
ENV_KEY = hashlib.sha256((PROJECT / "requirements-colab.txt").read_bytes()).hexdigest()[:12]
ENV_CACHE = PERSIST / ("environment-py312-" + ENV_KEY)
MODEL_DIR = PERSIST / "models/MiniMax-H3"
PY = PROJECT / ".venv/bin/python"

In [ ]:
# 可选：Colab 左侧钥匙图标中新建 HF_TOKEN，并授权当前笔记本。
# 不要把 token 粘贴进代码、配置或 GitHub。公开权重可不填。
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    pass

## 1. 查看下载清单
这一步只读随仓库提交的模型清单，无须安装依赖。

In [ ]:
run(sys.executable, "scripts/h3.py", "plan", "--workflow", WORKFLOW)

## 2. 缓存环境
下载 CUDA 12.8 的 PyTorch 轮子，并构建固定提交的 Diffusers；不会编译 CUDA 内核。随后在 CPU 上检查 H3 API 和 MP4 音画封装。首次耗时较长；完成标记存在时跳过重复构建。

In [ ]:
if not (ENV_CACHE / "environment.json").is_file():
    run(sys.executable, "scripts/environment.py", "cache", "--cache", ENV_CACHE)
else:
    print("已有环境缓存；如需重新解析依赖，请显式重跑 cache 命令。")
run(sys.executable, "scripts/environment.py", "install", "--cache", ENV_CACHE)

## 3. 下载并检查模型
只下载选定任务的 Diffusers 组件，不下载另一套 transformer 或原始 FL2VA/Ref2VA 副本。先确认已阅读模型许可；若 Hugging Face 要求授权，请先在模型页完成。

In [ ]:
run(PY, "scripts/h3.py", "download", "--workflow", WORKFLOW, "--model-dir", MODEL_DIR)
run(PY, "scripts/h3.py", "verify", "--workflow", WORKFLOW, "--model-dir", MODEL_DIR)

## 4. 准备完成
记下上方代码提交 SHA。结束 CPU 会话，在 GPU 笔记本设置相同仓库、提交和 Drive 路径。
配置和提示词尽量在本地修改后推到 GitHub。权重、安装包、token 和私人输入素材不要上传 GitHub。

In [ ]:
print("准备完成。代码提交:", COMMIT, "\nDrive:", PERSIST)